# 01 — Vegetation time-series analysis: NDVI, EVI2, Savitzky–Golay, PELT
NASA HLS and ESA Sentinel-2 observations are quality-masked and **only real acquisition dates** count. Retrospective spectral green-up is not independently verified crop emergence. PELT segmentation requires the Python `ruptures` package included in `pyproject.toml`.

**BFAST is not implemented:** full additive season/trend breakpoint analysis is more appropriate after obtaining dense multi-year vegetation time series. The official BFAST implementation is an R package.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd()
assert (ROOT/'sources.json').exists(), 'Run Jupyter in the Agrisimfinal project root'
BATCH_ROOT=ROOT/'data/processed/evia_sowing/hls_batches'
BATCHES=sorted(p for p in BATCH_ROOT.glob('*') if p.is_dir())
print('Batches found:',len(BATCHES))
BATCHES[-3:]

## 1. Build satellite vegetation indices and the original baseline
Indices are calculated from quality-masked reflectance. A provisional 3×3 sampling window is **not** a surveyed parcel. To use confirmed field geometries, supply the `parcels` argument to `build_vegetation`.


In [ ]:
from agrisim.timeseries.vegetation_indices import build as build_vegetation
from agrisim.timeseries.greenup import run as build_phenology
if BATCHES:
    observations=build_vegetation(hls_batch=BATCHES[-1])
    events=build_phenology()
    display(events)
else:
    print('No HLS batches yet; run the approved NASA downloader, then rerun this cell.')

## 2. Savitzky–Golay smoothing and PELT change-point detection
The existing baseline runs above. Now run **real** PELT (`ruptures.Pelt`) on smoothed NDVI and EVI2, then validate nominated breaks against **original** real observations. Daily interpolation is used **only internally for smoothing** and cannot create new satellite observations. Gaps longer than 40 days are never interpolated across.

PELT is not guaranteed to find green-up; `no_confirmed_greenup` is an acceptable result. The penalty (`3.0`) is a starting hyperparameter, **not** a validated threshold.


In [ ]:
from agrisim.timeseries.greenup import run as detect_greenup, SERIES_OUTPUT, OUTPUT
PELT_OUTPUT = ROOT / "data/processed/features/greenup_events_pelt.csv"
SMOOTHED_OUTPUT = ROOT / "data/processed/features/vegetation_smoothed.csv"
if (ROOT/"data/processed/features/vegetation_observations.csv").exists():
    try:
        pelt_events = detect_greenup(method="pelt", out=PELT_OUTPUT,
            pelt_penalty=3.0, savgol_window_days=15, series_out=SMOOTHED_OUTPUT)
        display(pelt_events)
    except RuntimeError as exc:
        print(exc)  # Install with `uv sync` and rerun this cell.
else:
    print("Download/process an approved HLS or Sentinel-2 series first.")


In [ ]:
if SMOOTHED_OUTPUT.exists() and PELT_OUTPUT.exists():
    sm = pd.read_csv(SMOOTHED_OUTPUT,parse_dates=["date_utc"])
    events_p = pd.read_csv(PELT_OUTPUT)
    for (field, season, source), part in sm.groupby(["field_id", "season", "source"]):
        fig, ax = plt.subplots(figsize=(11,4))
        part = part.sort_values("date_utc")
        for raw, smooth, color in [("ndvi_median","ndvi_smoothed","tab:green"),
                                  ("evi2_median","evi2_smoothed","tab:orange")]:
            ax.scatter(part.date_utc,part[raw],color=color,s=25,label=raw)
            ax.plot(part.date_utc,part[smooth],color=color,alpha=.7,label=smooth)
        match=events_p[(events_p.field_id==field)&(events_p.season==season)&(events_p.source==source)]
        if not match.empty and match.iloc[0].status=="retrospective_spectral_greenup":
            e=match.iloc[0]
            ax.axvspan(pd.to_datetime(e.last_low_date),pd.to_datetime(e.first_high_date),
                       color="grey",alpha=.15,label="Transition interval")
        ax.set(title=f"{field} / {season} / {source}: PELT + Savitzky–Golay",ylabel="Index")
        ax.legend(ncol=3,fontsize="small")
        plt.show()
else:
    print("PELT and smoothing outputs are not available yet.")


In [ ]:
path=ROOT/'data/processed/features/vegetation_observations.csv'
if path.exists():
    obs=pd.read_csv(path,parse_dates=['date_utc'])
    for field,part in obs.groupby('field_id'):
        ax=part.set_index('date_utc')[['ndvi_median','evi2_median']].sort_index().plot(title=f'{field} | reflective vegetation indices')
        ax.set_ylabel('Index');plt.show()
else:
    print('No observations yet')

**Interpretation:** this is spectral greening, not crop-emergence ground truth. Images of multiple dates may be cloudy; the algorithm retains a bracketed interval rather than inventing an exact day.